In [21]:

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

print("NeuroVision AI 2.0 Ready!")


NeuroVision AI 2.0 Ready!


In [22]:

noise = widgets.FloatSlider(
    value=0.25, min=0.05, max=0.50, step=0.05,
    description="Dataset Noise:"
)
neurons = widgets.IntSlider(
    value=32, min=8, max=64, step=8,
    description="Neurons:"
)
layers = widgets.IntSlider(
    value=2, min=1, max=4, step=1,
    description="Hidden Layers:"
)
epochs = widgets.IntSlider(
    value=300, min=100, max=700, step=100,
    description="Training Iterations:"
)
button = widgets.Button(
    description="Train NeuroVision AI",
    button_style="success"
)
output = widgets.Output()

display(widgets.VBox([noise, neurons, layers, epochs, button, output]))


In [23]:

def run(_):
    global scaler, improved
    with output:
        clear_output(wait=True)

        X, y = make_moons(
            n_samples=1000, noise=noise.value, random_state=42
        )
        Xtr, Xte, ytr, yte = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )

        scaler = StandardScaler()
        Xtr = scaler.fit_transform(Xtr)
        Xte = scaler.transform(Xte)

        basic = MLPClassifier(
            hidden_layer_sizes=(16,), max_iter=1500,
            random_state=42
        )
        improved = MLPClassifier(
            hidden_layer_sizes=(neurons.value,) * layers.value,
            max_iter=epochs.value, random_state=42
        )

        basic.fit(Xtr, ytr)
        improved.fit(Xtr, ytr)

        for name, model in [("Basic ANN", basic), ("Improved ANN", improved)]:
            pred = model.predict(Xte)
            print(f"{name}: Accuracy = {accuracy_score(yte, pred)*100:.2f}%"
                  f" | F1-score = {f1_score(yte, pred):.3f}")

        visualize(X, y, Xte, yte, scaler, basic, improved)

button.on_click(run)


In [24]:

def visualize(X, y, Xte, yte, scaler, basic, improved):
    xx, yy = np.meshgrid(
        np.linspace(-2, 3, 180), np.linspace(-1.5, 2, 180)
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    prob = improved.predict_proba(scaler.transform(grid))[:, 1]
    prob = prob.reshape(xx.shape)

    fig, ax = plt.subplots(1, 3, figsize=(16, 4))

    # 1. Confidence map
    ax[0].contourf(xx, yy, prob, levels=20, cmap="coolwarm")
    ax[0].contour(xx, yy, prob, levels=[0.4, 0.5, 0.6],
                  colors=["gold", "black", "gold"])
    ax[0].scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm",
                  edgecolors="black", s=10)
    ax[0].set_title("1. Confidence Map")

    # 2. ANN battle
    names = ["Basic ANN", "Improved ANN"]
    scores = [
        accuracy_score(yte, basic.predict(Xte)) * 100,
        accuracy_score(yte, improved.predict(Xte)) * 100
    ]
    ax[1].bar(names, scores)
    ax[1].set_ylim(0, 100)
    ax[1].set_title("2. ANN Battle")
    ax[1].set_ylabel("Accuracy (%)")

    # 3. Noise challenge
    levels = [0.1, 0.2, 0.3, 0.4, 0.5]
    for name, model in zip(names, [basic, improved]):
        scores = []
        for n in levels:
            a, b = make_moons(n_samples=500, noise=n, random_state=7)
            scores.append(
                accuracy_score(b, model.predict(scaler.transform(a))) * 100
            )
        ax[2].plot(levels, scores, marker="o", label=name)

    ax[2].set_title("3. Noise Challenge")
    ax[2].set_xlabel("Noise level")
    ax[2].set_ylabel("Accuracy (%)")
    ax[2].legend()

    plt.tight_layout()
    plt.show()

    # Extra: uncertainty report
    uncertain = ((prob >= 0.4) & (prob <= 0.6)).mean() * 100
    print(f"Uncertain area of plotted grid: {uncertain:.2f}%")
    print("Experiment complete!")


In [25]:

# Change these coordinates to test different points
x1, x2 = 0.5, 0.2

# Run Cell 3 first, then use the trained model and scaler
point = scaler.transform([[x1, x2]])
probabilities = improved.predict_proba(point)[0]

print("Predicted class:", np.argmax(probabilities))
print(f"Model confidence: {max(probabilities)*100:.2f}%")


Predicted class: 1
Model confidence: 56.82%
